In [12]:
%pip install python-dotenv rank-bm25 sentence-transformers langchain langchain-community langchain-huggingface langchain-google-genai numpy --quiet

In [13]:
import os
import getpass
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Use getpass to hide your key from the notebook output
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

print("Setup complete.")

Setup complete.


In [14]:
# Expanded corpus to include technical terms for BM25 and enough content for chunking
corpus = [
    "Transformers use self-attention mechanisms to process sequences in parallel.",
    "The attention mechanism computes a weighted sum of value vectors based on query-key similarity.",
    "BERT is a bidirectional encoder trained using masked language modelling.",
    "The BM25 algorithm ranks documents based on term frequency and inverse document frequency.",
    "Gradient descent is an optimization technique used to minimize the loss function.",
    "The AdamW optimizer is a popular variant of gradient descent that handles weight decay better.",
    "Neural networks learn by adjusting weights through backpropagation.",
    "Quantization reduces model size by representing weights in lower bit formats such as INT8.",
    "Large language models are trained on massive text corpora to learn general-purpose representations.",
    "Fine-tuning adapts a pre-trained model to a specific downstream task using task-specific data.",
    "Stochastic Gradient Descent (SGD) updates parameters using a single random training example at a time.",
    "The learning rate is a hyperparameter that controls how much to change the model in response to the estimated error each time the model weights are updated."
]

print(f"Corpus: {len(corpus)} documents loaded.")

Corpus: 12 documents loaded.


In [15]:
class AdvancedHybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k

        # 1. BM25 Setup
        tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized_corpus)

        # 2. SBERT Setup
        self.sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        self.doc_embeddings = self.sbert.encode(corpus, convert_to_numpy=True)
        self.doc_embeddings = self.doc_embeddings / np.linalg.norm(self.doc_embeddings, axis=1, keepdims=True)

    def colbert_score(self, query: str, document: str) -> float:
        # Bonus 3: ColBERT MaxSim implementation
        q_tokens = query.lower().split()
        d_tokens = document.lower().split()
        q_vecs = self.sbert.encode(q_tokens)
        d_vecs = self.sbert.encode(d_tokens)
        q_vecs /= np.linalg.norm(q_vecs, axis=1, keepdims=True)
        d_vecs /= np.linalg.norm(d_vecs, axis=1, keepdims=True)
        sim_matrix = q_vecs @ d_vecs.T
        return float(sim_matrix.max(axis=1).sum())

    def retrieve(self, query: str, top_k: int = 5, alpha: float = 0.5) -> list[dict]:
        # 1. BM25 Ranking
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_ranked)}

        # 2. SBERT Ranking
        q_vec = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec /= np.linalg.norm(q_vec)
        sbert_scores = self.doc_embeddings @ q_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_ranked)}

        # 3. Bonus 3: ColBERT Ranking
        colbert_scores = [self.colbert_score(query, doc) for doc in self.corpus]
        colbert_ranked = np.argsort(colbert_scores)[::-1]
        colbert_ranks = {doc_id: rank + 1 for rank, doc_id in enumerate(colbert_ranked)}

        # 4. RRF Fusion (Fusing all 3 lists)
        results = []
        for i in range(len(self.corpus)):
            # We add the ColBERT rank as a third factor in the democratic RRF sum
            rrf_score = (alpha * (1 / (self.k + bm25_ranks[i]))) + \
                        ((1 - alpha) * (1 / (self.k + sbert_ranks[i]))) + \
                        (1 / (self.k + colbert_ranks[i])) # 3rd rank added here
            results.append({
                "doc_id": i, "rrf_score": rrf_score,
                "bm25_rank": bm25_ranks[i], "sbert_rank": sbert_ranks[i],
                "colbert_rank": colbert_ranks[i], "text": self.corpus[i]
            })
        return sorted(results, key=lambda x: x["rrf_score"], reverse=True)[:top_k]

retriever = AdvancedHybridRetriever(corpus)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_k=3):
    pairs = [[query, c["text"]] for c in candidates]
    scores = cross_encoder.predict(pairs)
    for i, s in enumerate(scores):
        candidates[i]["ce_score"] = s
    return sorted(candidates, key=lambda x: x["ce_score"], reverse=True)[:top_k]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)
hyde_prompt = ChatPromptTemplate.from_template("Write a factual paragraph about: {query}")
hyde_chain = hyde_prompt | llm | StrOutputParser()

In [18]:
def advanced_rag(user_query: str, alpha: float = 0.5):
    # 1. Expansion
    hypo_doc = hyde_chain.invoke({"query": user_query})
    # 2. Hybrid Weighted Retrieval
    candidates = retriever.retrieve(hypo_doc, top_k=6, alpha=alpha)
    # 3. Re-Ranking
    top_docs = rerank(user_query, candidates, top_k=3)
    # 4. Answer Generation
    context = "\n\n".join([d["text"] for d in top_docs])
    chain = ChatPromptTemplate.from_template("Context: {context}\nQuestion: {question}") | llm | StrOutputParser()
    return chain.invoke({"context": context, "question": user_query})

print("Advanced RAG function ready.")

Advanced RAG function ready.


In [19]:
def chunk_size_study():
    long_text = " ".join(corpus * 5) # Create a synthetic long doc
    sizes = [10, 30, 60]

    print("Chunk Size Study Results:")
    for s in sizes:
        words = long_text.split()
        chunks = [" ".join(words[i:i+s]) for i in range(0, len(words), s)]
        temp_retriever = AdvancedHybridRetriever(chunks)
        results = temp_retriever.retrieve("attention in transformers", top_k=1)
        print(f" Size {s}: Found {len(chunks)} chunks. Top result length: {len(results[0]['text'].split())} words.")

chunk_size_study()

Chunk Size Study Results:


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Size 10: Found 82 chunks. Top result length: 10 words.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Size 30: Found 28 chunks. Top result length: 30 words.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Size 60: Found 14 chunks. Top result length: 60 words.


In [20]:
def naive_rag_top(query):
    q_vec = retriever.sbert.encode([query], convert_to_numpy=True)[0]
    q_vec /= np.linalg.norm(q_vec)
    scores = retriever.doc_embeddings @ q_vec
    return corpus[np.argsort(scores)[::-1][0]]

In [21]:
# Updated Helper to get the Top Doc from the Advanced Pipeline
def advanced_rag_top_doc(user_query: str, alpha: float = 0.5):
    # 1. Expand query with HyDE
    hypo_doc = hyde_chain.invoke({"query": user_query})
    # 2. Retrieve with Hybrid (BM25 + SBERT + ColBERT)
    candidates = retriever.retrieve(hypo_doc, top_k=6, alpha=alpha)
    # 3. Re-Rank with Cross-Encoder and return ONLY the top text
    top_docs = rerank(user_query, candidates, top_k=1)
    return top_docs[0]['text']

queries = ["how do transformers encode meaning?", "optimization techniques for training", "AdamW optimizer details"]

print("| Query | Naive Top Doc | Advanced Top Doc | Different? |")
print("|---|---|---|---|")
for q in queries:
    n = naive_rag_top(q)[:45] + "..."
    a = advanced_rag_top_doc(q)[:45] + "..."
    print(f"| {q} | {n} | {a} | {'Yes' if n != a else 'No'} |")

| Query | Naive Top Doc | Advanced Top Doc | Different? |
|---|---|---|---|
| how do transformers encode meaning? | Transformers use self-attention mechanisms to... | Transformers use self-attention mechanisms to... | No |
| optimization techniques for training | Neural networks learn by adjusting weights th... | Gradient descent is an optimization technique... | Yes |
| AdamW optimizer details | The AdamW optimizer is a popular variant of g... | The AdamW optimizer is a popular variant of g... | No |
